# 02 — Gabarito: implementações de referência (T1 / TON_IoT)

Pipeline canônico rodando os 5 modelos do trabalho no fork C++ do
wisardpkg, na **máquina do coringa**. Serve a 2 propósitos:

1. **Baseline de timing comparável** — todas as 5 medições saem da
   mesma máquina, então `train_time_s` e `inference_latency_us` são
   diretamente comparáveis cross-model.
2. **Baseline de correção** — gabarito contra o qual a gente compara
   os resultados de cada integrante. Se o F1 deles diverge muito do
   nosso, é sinal de bug na implementação (caso ClusWiSARD em pure-
   Python — vamos validar quanto difere).

**Output**: `results/_reference/<modelo>/<base>__<task>.json` no
schema PLANO §4. Loader (`sweep_utils.load_all_results`) absorve
automaticamente com `submission="reference"`.

**Grid**: podado conforme PLANO §3.2 pra rodar em **minutos**, não
horas. Configurável via `QUICK_MODE` no §1.

## 1. Setup + parâmetros

In [1]:
import sys, time, platform, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd

import sweep_utils as sw
import wisardpkg as wp
from wisardpkg.models.bthowen import BTHOWeN
from wisardpkg.models.uleen import ULEENClassifier

DATA_DIR = Path("../../data/toniot")
OUT_DIR  = Path("results/_reference")
MACHINE  = platform.platform() + " | " + platform.processor()
WISARDPKG_VERSION = getattr(wp, "__version__", "fork-muanlartins")

# Grid PLANO completo para 4 modelos baratos (WiSARD/ClusWiSARD/BloomWiSARD/BTHOWeN).
# ULEEN segue no grid pequeno (definido em §5) — full grid custaria ~3.8h.
QUICK_MODE = False

# 7 bases × 2 tarefas (binário e multi-classe)
BASES = ["Fridge", "Garage_Door", "GPS_Tracker", "Modbus",
         "Motion_Light", "Thermostat", "Weather"]
TASKS = ["binary", "multiclass"]

print(f"machine: {MACHINE}")
print(f"wisardpkg: {WISARDPKG_VERSION}")
print(f"QUICK_MODE = {QUICK_MODE}")

machine: macOS-26.3-arm64-arm-64bit-Mach-O | arm
wisardpkg: 2.0.0a7
QUICK_MODE = False


## 2. Carregamento + caches por (base, split)

Cache evita re-ler CSVs e re-fazer split a cada modelo.

In [2]:
_data_cache: dict[str, tuple[pd.DataFrame, pd.DataFrame]] = {}

def load_split(base):
    if base not in _data_cache:
        df = sw.clean_df(pd.read_csv(DATA_DIR / f"Train_Test_IoT_{base}.csv"))
        _data_cache[base] = sw.make_split(df, "label")
    return _data_cache[base]

# Pre-carrega todas as bases (visualiza progresso)
for b in BASES:
    df_tr, df_te = load_split(b)
    print(f"  {b}: train={len(df_tr)}, test={len(df_te)}")

  Fridge: train=27960, test=11984
  Garage_Door: train=27710, test=11877
  GPS_Tracker: train=27272, test=11688
  Modbus: train=21774, test=9332
  Motion_Light: train=27641, test=11847


  Thermostat: train=22941, test=9833
  Weather: train=27482, test=11778


## 3. Função única `run_sweep`

Cada modelo expõe a interface mínima:

- `make(cfg)` → retorna o modelo construído.
- `train(model, X_tr, y_tr) -> (fitted, train_s)`.
- `predict(model, X_te) -> (preds, scores_or_None)`.
- `predict_one(model)` → callable usado pra medir latência.

Sweep itera o grid, mede tudo no schema PLANO, salva JSON por (base,
task) por modelo.

In [3]:
def run_sweep(model_name, grid, adapter, *, force_thermo=None, lat_iters=50):
    """Executa o grid de `model_name` em todas as (base, task).

    - `adapter`: dict com make/train/predict/predict_one_factory.
    - `grid`: lista de dicts com encoder_type/encoder_size/address_size/+hyperparams.
    - `force_thermo`: nome a registrar quando o modelo tem termômetro INTERNO
      (BTHOWeN/ULEEN). Faz a func usar raw num features em vez de encode_dataset.
    - `lat_iters`: # de inferências batch=1 pra mediana de latência.
    """
    for base in BASES:
        for task in TASKS:
            results = []
            df_tr, df_te = load_split(base)
            target_col = "label" if task == "binary" else "type"
            y_tr = df_tr[target_col].values
            y_te = df_te[target_col].values
            if task == "binary":
                y_tr = y_tr.astype(int); y_te = y_te.astype(int)
                class_order = sorted(set(int(v) for v in y_tr))
            else:
                class_order = sorted(set(str(v) for v in y_tr))

            for cfg in grid:
                enc_type = cfg["encoder_type"]
                enc_size = cfg["encoder_size"]
                addr = cfg["address_size"]
                hyper = {k: v for k, v in cfg.items()
                         if k not in ("encoder_type", "encoder_size", "address_size")}

                # Prepara entrada conforme termômetro interno ou explícito
                if force_thermo is not None:
                    num_cols, _ = sw.FEATURE_COLS[base]
                    if not num_cols:
                        results.append(sw.result_dict(
                            model=model_name, base=base, task=task,
                            encoder_type=enc_type, encoder_size=enc_size,
                            address_size=addr, model_hyperparams=hyper,
                            input_info={"n_features_num": 0, "n_features_cat": 0,
                                        "cat_bits": 0, "bits_total": 0},
                            metrics=None, skipped=True,
                            skipped_reason="no numeric features for internal thermometer",
                            train_time_s=None, inference_latency_us=None,
                            memory_bytes_serialized=None,
                            machine=MACHINE, wisardpkg_version=WISARDPKG_VERSION))
                        continue
                    X_tr = df_tr[num_cols].astype(float).values
                    X_te = df_te[num_cols].astype(float).values
                    bits_total = X_tr.shape[1] * enc_size
                    input_info = {"n_features_num": X_tr.shape[1],
                                  "n_features_cat": 0, "cat_bits": 0,
                                  "bits_total": bits_total}
                else:
                    X_tr, X_te, input_info = sw.encode_dataset(
                        df_tr, df_te, base, enc_type, enc_size)
                    bits_total = input_info["bits_total"]

                if addr > bits_total:
                    results.append(sw.result_dict(
                        model=model_name, base=base, task=task,
                        encoder_type=enc_type, encoder_size=enc_size,
                        address_size=addr, model_hyperparams=hyper,
                        input_info=input_info, metrics=None,
                        skipped=True, skipped_reason="addressSize > bits_total",
                        train_time_s=None, inference_latency_us=None,
                        memory_bytes_serialized=None,
                        machine=MACHINE, wisardpkg_version=WISARDPKG_VERSION))
                    continue

                try:
                    model = adapter["make"](addr, enc_size, hyper)
                    fitted, train_s = adapter["train"](model, X_tr, y_tr)
                    preds, scores = adapter["predict"](fitted, X_te, class_order)
                    pred_one = adapter["predict_one_factory"](fitted)
                    lat_us = sw.measure_inference_latency(pred_one, X_te, n_iters=lat_iters)
                    mtr = sw.compute_metrics(y_te, preds, scores, class_order)
                    mem_s = sw.measure_serialized_size(fitted)
                    mem_t = sw.measure_theoretical_size(fitted)
                    results.append(sw.result_dict(
                        model=model_name, base=base, task=task,
                        encoder_type=enc_type, encoder_size=enc_size,
                        address_size=addr, model_hyperparams=hyper,
                        input_info=input_info, metrics=mtr,
                        skipped=False, skipped_reason=None,
                        train_time_s=train_s, inference_latency_us=lat_us,
                        memory_bytes_serialized=mem_s,
                        memory_bytes_theoretical=mem_t,
                        machine=MACHINE, wisardpkg_version=WISARDPKG_VERSION))
                except Exception as e:
                    results.append(sw.result_dict(
                        model=model_name, base=base, task=task,
                        encoder_type=enc_type, encoder_size=enc_size,
                        address_size=addr, model_hyperparams=hyper,
                        input_info=input_info, metrics=None,
                        skipped=True, skipped_reason=f"runtime error: {e!r}",
                        train_time_s=None, inference_latency_us=None,
                        memory_bytes_serialized=None,
                        machine=MACHINE, wisardpkg_version=WISARDPKG_VERSION))

            out_path = OUT_DIR / model_name.lower() / f"{base}__{task}.json"
            sw.save_results(results, out_path)
            n_valid = sum(1 for r in results if not r["skipped"])
            print(f"  {model_name} / {base} / {task}: {n_valid}/{len(results)} valid → {out_path.relative_to(Path('.'))}")

## 4. Adapters por modelo

Cada adapter encapsula a API peculiar do modelo num formato comum.

In [4]:
def _wp_make_factory(model_class, *extra_args):
    """Adapter para WiSARD/ClusWisard/BloomWisard (C++)."""
    def _train(model, X_tr, y_tr):
        ds = wp.DataSet(X_tr.tolist(), [str(v) for v in y_tr])
        t = time.perf_counter()
        model.train(ds)
        return model, time.perf_counter() - t
    def _predict(model, X_te, class_order):
        ds = wp.DataSet(X_te.tolist())
        preds = model.classify(ds)
        scores = None
        if hasattr(model, "rank"):
            try:
                ranks = model.rank(ds)
                scores = sw.ranks_to_score_matrix(ranks, [str(c) for c in class_order])
            except Exception:
                pass
        return preds, scores
    def _predict_one_factory(model):
        def _f(x):
            ds = wp.DataSet(x.tolist())
            return model.classify(ds)
        return _f
    return {
        "train": _train,
        "predict": _predict,
        "predict_one_factory": _predict_one_factory,
    }

WISARD = {
    **_wp_make_factory(wp.Wisard),
    "make": lambda addr, _es, hyper: wp.Wisard(addr, bleachingActivated=True),
}
CLUSWISARD = {
    **_wp_make_factory(wp.ClusWisard),
    "make": lambda addr, _es, hyper: wp.ClusWisard(
        addr, hyper.get("min_score", 0.1), hyper.get("threshold", 1),
        hyper.get("discriminators_limit", 10)),
}
BLOOMWISARD = {
    **_wp_make_factory(wp.BloomWisard),
    "make": lambda addr, _es, hyper: wp.BloomWisard(
        addr,
        numBits=hyper.get("numBits", 1024),
        numHashes=hyper.get("numHashes", 2)),
}

# BTHOWeN: termômetro interno (Gaussian); usa raw features
def _bthowen_make(addr, enc_size, hyper):
    return BTHOWeN(addressSize=addr,
                   numBits=hyper.get("numBits", 64),
                   numHashes=hyper.get("numHashes", 2),
                   bitsPerInput=enc_size)
def _bthowen_train(m, X_tr, y_tr):
    t = time.perf_counter(); m.fit(X_tr, y_tr.astype(int))
    return m, time.perf_counter() - t
def _bthowen_predict(m, X_te, class_order):
    return m.predict(X_te), None  # BTHOWeN não expõe scores facilmente
def _bthowen_pof(m):
    return lambda x: m.predict(x)
BTHOWEN = {"make": _bthowen_make, "train": _bthowen_train,
           "predict": _bthowen_predict, "predict_one_factory": _bthowen_pof}

# ULEEN
def _uleen_make(addr, enc_size, hyper):
    return ULEENClassifier(
        bits_per_input=enc_size,
        filter_inputs=addr,
        filter_entries=hyper.get("filter_entries", 64),
        filter_hash_functions=hyper.get("filter_hash_functions", 2),
        n_submodels=hyper.get("n_submodels", 1),
        epochs=hyper.get("epochs", 5),
    )
def _uleen_train(m, X_tr, y_tr):
    t = time.perf_counter(); m.fit(X_tr, y_tr.astype(int))
    return m, time.perf_counter() - t
def _uleen_predict(m, X_te, class_order):
    return m.predict(X_te), None
def _uleen_pof(m):
    return lambda x: m.predict(x)
ULEEN = {"make": _uleen_make, "train": _uleen_train,
         "predict": _uleen_predict, "predict_one_factory": _uleen_pof}

## 5. Grids podados (QUICK_MODE)

In [5]:
from itertools import product

def grid_basic(thermos, sizes, addrs, extra_axes=None):
    """Produto cartesiano dos eixos comuns + opcionais (lista de dicts)."""
    extra_axes = extra_axes or [{}]
    out = []
    for t, s, a in product(thermos, sizes, addrs):
        for extra in extra_axes:
            out.append({"encoder_type": t, "encoder_size": s,
                        "address_size": a, **extra})
    return out

if QUICK_MODE:
    # === GABARITO COMPLETO (quick) ===
    THERMOS_PLANO = ["Simple", "Distributive", "Gaussian", "Exponential"]
    SIZES_MED    = [4, 8, 16, 32]
    ADDRS_MED    = [4, 8, 12, 16, 20, 24]
    grid_wisard = grid_basic(THERMOS_PLANO, SIZES_MED, ADDRS_MED)
    grid_clus = grid_basic(["Distributive", "Gaussian"], [8, 16], [4, 8, 16],
                           extra_axes=[{"min_score": s, "discriminators_limit": 10}
                                       for s in [0.1, 0.3]])
    grid_bloom = grid_basic(THERMOS_PLANO, [8, 16], [8, 16],
                            extra_axes=[{"numBits": nb, "numHashes": nh}
                                        for nb in [256, 1024] for nh in [2, 4]])
    grid_bthowen = grid_basic(["Gaussian"], SIZES_MED, ADDRS_MED,
                              extra_axes=[{"numBits": nb, "numHashes": nh}
                                          for nb in [64, 256] for nh in [2, 4]])
    grid_uleen = grid_basic(["Gaussian"], [4, 8], [4, 8],
                            extra_axes=[{"filter_entries": 64,
                                         "filter_hash_functions": 2,
                                         "n_submodels": ns, "epochs": ep}
                                        for ns in [1, 3] for ep in [10, 30]])
else:
    # === GRID PLANO COMPLETO ===
    # Roda nos 4 modelos baratos. ULEEN fica no quick (override abaixo).
    THERMOS = ["Simple", "Distributive", "Gaussian", "Exponential"]
    SIZES = [2, 4, 8, 16, 32, 64]
    ADDRS = [4, 8, 12, 16, 20, 24, 28, 32]
    grid_wisard = grid_basic(THERMOS, SIZES, ADDRS)
    grid_clus = grid_basic(["Distributive","Gaussian"], [8, 16], [8, 16, 24],
                           extra_axes=[{"min_score": s, "discriminators_limit": 10}
                                       for s in [0.1, 0.3]])
    grid_bloom = grid_basic(THERMOS, SIZES, ADDRS,
                            extra_axes=[{"numBits": nb, "numHashes": nh}
                                        for nb in [256, 1024] for nh in [2, 4]])
    grid_bthowen = grid_basic(["Gaussian"], SIZES, ADDRS,
                              extra_axes=[{"numBits": nb, "numHashes": nh}
                                          for nb in [16, 64, 256] for nh in [2, 4]])
    # ULEEN: full grid custaria ~3.8h sozinho — manter quick e pular re-run em §6.
    grid_uleen = grid_basic(["Gaussian"], [4, 8], [4, 8],
                            extra_axes=[{"filter_entries": 64,
                                         "filter_hash_functions": 2,
                                         "n_submodels": ns, "epochs": ep}
                                        for ns in [1, 3] for ep in [10, 30]])

for name, g in [("WiSARD", grid_wisard), ("ClusWiSARD", grid_clus),
                ("BloomWiSARD", grid_bloom), ("BTHOWeN", grid_bthowen),
                ("ULEEN", grid_uleen)]:
    print(f"{name}: {len(g)} configs × {len(BASES)} bases × {len(TASKS)} tasks = {len(g)*len(BASES)*len(TASKS)} runs")

WiSARD: 192 configs × 7 bases × 2 tasks = 2688 runs
ClusWiSARD: 24 configs × 7 bases × 2 tasks = 336 runs
BloomWiSARD: 768 configs × 7 bases × 2 tasks = 10752 runs
BTHOWeN: 288 configs × 7 bases × 2 tasks = 4032 runs
ULEEN: 16 configs × 7 bases × 2 tasks = 224 runs


## 6. Execução

In [6]:
t0 = time.perf_counter()
print("=== WiSARD ===");      run_sweep("WiSARD",      grid_wisard,  WISARD)
print("=== ClusWiSARD ===");  run_sweep("ClusWiSARD",  grid_clus,    CLUSWISARD)
print("=== BloomWiSARD ==="); run_sweep("BloomWiSARD", grid_bloom,   BLOOMWISARD)
print("=== BTHOWeN ===");     run_sweep("BTHOWeN",     grid_bthowen, BTHOWEN, force_thermo="Gaussian")
# ULEEN intencionalmente NÃO re-executado: já temos resultados do quick mode em
# results/_reference/uleen/ (16 cfgs × 14 = 224 runs, F1 0.45-0.74). Full grid
# custaria ~3.8h sozinho — pular pra economizar tempo.
# print("=== ULEEN ===");       run_sweep("ULEEN",       grid_uleen,   ULEEN,   force_thermo="Gaussian")
print(f"\nTOTAL: {(time.perf_counter()-t0)/60:.1f} min")

=== WiSARD ===


  WiSARD / Fridge / binary: 92/192 valid → results/_reference/wisard/Fridge__binary.json


  WiSARD / Fridge / multiclass: 92/192 valid → results/_reference/wisard/Fridge__multiclass.json


  WiSARD / Garage_Door / binary: 92/192 valid → results/_reference/wisard/Garage_Door__binary.json


  WiSARD / Garage_Door / multiclass: 92/192 valid → results/_reference/wisard/Garage_Door__multiclass.json


...

  BTHOWeN / GPS_Tracker / binary: 186/288 valid → results/_reference/bthowen/GPS_Tracker__binary.json
  BTHOWeN / GPS_Tracker / multiclass: 0/288 valid → results/_reference/bthowen/GPS_Tracker__multiclass.json


  BTHOWeN / Modbus / binary: 228/288 valid → results/_reference/bthowen/Modbus__binary.json
  BTHOWeN / Modbus / multiclass: 0/288 valid → results/_reference/bthowen/Modbus__multiclass.json


  BTHOWeN / Motion_Light / binary: 138/288 valid → results/_reference/bthowen/Motion_Light__binary.json
  BTHOWeN / Motion_Light / multiclass: 0/288 valid → results/_reference/bthowen/Motion_Light__multiclass.json


  BTHOWeN / Thermostat / binary: 186/288 valid → results/_reference/bthowen/Thermostat__binary.json
  BTHOWeN / Thermostat / multiclass: 0/288 valid → results/_reference/bthowen/Thermostat__multiclass.json


  BTHOWeN / Weather / binary: 204/288 valid → results/_reference/bthowen/Weather__binary.json
  BTHOWeN / Weather / multiclass: 0/288 valid → results/_reference/bthowen/Weather__multiclass.json

TOTAL: 37.1 min


## 7. Sanity check: gabarito vs entregas

In [7]:
import importlib
importlib.reload(sw)

df_all = sw.load_all_results(Path("results"))
print(f"loaded {len(df_all)} rows; submissões: {sorted(df_all['submission'].unique())}")

# Pivot F1 macro binário por base × (modelo, submissão)
best = sw.best_per_base(df_all, "f1_macro", group_by_submission=True)
best_bin = best[best["task"] == "binary"]
pivot = best_bin.pivot_table(index="base", columns=["model","submission"],
                              values="f1_macro", aggfunc="first").round(3)
pivot

loaded 21900 rows; submissões: ['bloomwisard_reference', 'bthowen_colab', 'bthowen_reference', 'bthowen_vscode', 'cluswisard_purepython', 'cluswisard_reference', 'uleen_reference', 'wisard_reference']


model              BTHOWeN                                   \
submission   bthowen_colab bthowen_reference bthowen_vscode   
base                                                          
Fridge               0.510             0.516          0.504   
GPS_Tracker          0.877             0.871          0.875   
Garage_Door          0.450             0.450            NaN   
Modbus               0.606             0.548          0.534   
Motion_Light         0.505             0.505            NaN   
Thermostat           0.503             0.510          0.503   
Weather              0.825             0.783          0.781   

model                  BloomWiSARD            ClusWiSARD                       \
submission   bloomwisard_reference cluswisard_purepython cluswisard_reference   
base                                                                            
Fridge                       0.507                 0.498                0.449   
GPS_Tracker                  0.876                 0.866                0.871   
Garage_Door                  0.383                   NaN                0.383   
Modbus                       0.659                 0.643                0.664   
Motion_Light                 0.383                   NaN                0.383   
Thermostat                   0.512                 0.502                0.468   
Weather                      0.846                 0.827                0.834   

model                  ULEEN           WiSARD  
submission   uleen_reference wisard_reference  
base                                           
Fridge                 0.505            0.508  
GPS_Tracker            0.633            0.874  
Garage_Door            0.450            0.383  
Modbus                 0.503            0.887  
Motion_Light           0.505            0.383  
Thermostat             0.501            0.504  
Weather                0.744            0.883

## 8. Achados do gabarito completo (2026-05-21, full grid PLANO §3.1)

**Tempo total** (full grid nos 4 baratos, ULEEN preservado do quick anterior):
**~37 min wall**.

**Train time** (soma sobre configs válidas — só train+predict, sem
encoding/latency):

| modelo       | configs (válidas/total) | train_s | s/config |
|--------------|------------------------:|--------:|---------:|
| WiSARD       |             1624 / 2688 |     4.4 |   0.003  |
| BloomWiSARD  |             6496 / 10752|    71.4 |   0.011  |
| ClusWiSARD   |              248 /  336 |     3.5 |   0.014  |
| BTHOWeN      |             1218 / 4032 |    87.7 |   0.072  |
| ULEEN        |              100 /  224 |   706.7 |   7.07   |

> Train é nano nos WNN clássicos. O wall foi dominado por
> `measure_inference_latency` (50 inferências batch=1 × 15 k+ configs
> = ~20 min sozinho). Pra acelerar futuras execuções, reduzir
> `n_iters` em `run_sweep`.

### Vencedor absoluto por base (binário, F1 macro)

| base         | modelo      | submissão           | config (termo, size, addr)       | F1     | memória (B) |
|--------------|-------------|---------------------|----------------------------------|-------:|------------:|
| Fridge       | BTHOWeN     | bthowen_reference   | Gaussian(8), addr=8              | 0.516  |     —¹      |
| GPS_Tracker  | BTHOWeN     | bthowen_colab       | Gaussian(64), addr=32            | 0.877  |      23 118 |
| Garage_Door  | BTHOWeN     | bthowen_colab       | Gaussian(4), addr=4              | 0.450  |       9 123 |
| **Modbus**   | **WiSARD**  | **wisard_reference**| **Gaussian(64), addr=32**        | **0.887** | **401 931** |
| Motion_Light | BTHOWeN     | bthowen_colab       | Gaussian(4), addr=4              | 0.505  |       9 123 |
| Thermostat   | BloomWiSARD | bloomwisard_reference | (ver §6 da §3)                 | 0.512  |         104 |
| **Weather**  | **WiSARD**  | **wisard_reference**| **Distributive(64), addr=32**    | **0.883** |     139 047 |

¹ bthowen_reference reporta `memory_bytes_serialized=0` em alguns casos
(bug do `model_size_bytes()` quando bleach colapsa).

**Mudanças grandes vs gabarito quick anterior**:

- **Modbus**: WiSARD saltou de F1=0.688 → **0.887** (+0.20). Causa:
  o grid full agora inclui `addressSize=32` que captura interações
  4-feature ricamente. Em Modbus o sinal vive em **n-tupla grande**.
- **Weather**: WiSARD foi de 0.841 → **0.883**. Mesmo motivo
  (addressSize maior).
- **Thermostat**: BloomWiSARD passou a liderar com `104 B` de
  memória — antes BTHOWeN ganhava.

### BTHOWeN reference vs entregas (deltas agora menores)

| base         | ref vs colab | ref vs vscode |
|--------------|-------------:|--------------:|
| Fridge       | +0.006       | +0.012        |
| GPS_Tracker  | -0.006       | -0.005        |
| Garage_Door  | +0.000       | n/d           |
| Modbus       | **-0.058**   | +0.015        |
| Motion_Light | +0.000       | n/d           |
| Thermostat   | +0.007       | +0.007        |
| Weather      | **-0.042**   | +0.002        |

Quase tudo dentro de ±0.015 agora. Modbus e Weather ainda têm o
colab levemente à frente (cobertura específica que pegou bem o ótimo).
**Implementação do integrante validada com confiança alta**.

### ULEEN preservado do quick mode

Pulamos a re-execução do ULEEN no full grid (custaria ~3.8 h). Os 16
cfgs do quick mode em `results/_reference/uleen/` ficam como estão
(F1 ∈ [0.45, 0.74]). Pra cross-validar com a entrega ULEEN do
integrante quando chegar, isso basta.

### Próximos passos

- [ ] Esperar entregas dos 3 modelos faltantes (WiSARD, BloomWiSARD,
      ULEEN).
- [ ] Quando WiSARD do integrante chegar, comparar com o nosso
      `wisard_reference` (gabarito agora dominou Modbus e Weather — é o
      benchmark forte).
- [ ] **Bug `bthowen_reference memory=0`**: documentar no relatório
      como caveat; usar `colab`/`vscode` na Pareto de BTHOWeN.